- [Isamlic QA Egyptian](https://huggingface.co/datasets/Omar-youssef/islamic-qa-egyptian-arabic)

- [Qwen3 - 8B](https://huggingface.co/unsloth/Qwen3-8B)

- [Unsloth Fine-Tuning](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide)

- [Quantization]()

### Setup & Imports

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret('HF_TOKEN')
WANDB_TOKEN = user_secrets.get_secret("WANDB_TOKEN")

!huggingface-cli login --token {HF_TOKEN}
!wandb login {WANDB_TOKEN}

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Fine-Tune-Summarization` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Fine-Tune-Summarization`
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key

In [3]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[bitsandbytes.cextension|ERROR]bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


ImportError: Support for Transformers v4 is deprecated and was removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers

### Load the Model

In [ ]:
model_id = 'unsloth/Qwen3-8B'
model, tokenizer = FastLanguageModel.from_pretrained(model_id,
                                                     load_in_4bit=True,
                                                     use_gradient_checkpointing='unsloth')

In [ ]:
# model

#### We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    "gate_proj", "up_proj", "down_proj",
                    ],
    lora_alpha=32,  # Best to choose alpha = rank or rank*2
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=3407,
    use_rslora=True,
    loftq_config=None
)

# model

### Dataset Preparation

In [ ]:
dataset = load_dataset("Omar-youssef/islamic-qa-egyptian-arabic")

dataset

# dataset = dataset['train'].train_test_split(test_size=0.1, seed=3407)
# train_ds = dataset['train']
# eval_ds = dataset['test']  # احجزه للتقييم بس

In [ ]:
# @title To Detect the Intent from the Source Topics (aqeedah, fiqh, history, general)

aqeedah_keywords = [
    # الله والتوحيد والعقيدة
    "الله",
    "التوحيد",
    "العقيدة",
    "الإيمان",
    "الايمان",
    "الإحسان",
    "الاحسان",
    "الشريعة",
    # الأركان والغيبات
    "القدر",
    "الملائكة",
    "الملائكه",
    "الكتب السماوية",
    "الكتب السماويه",
    "الرسل",
    "الأنبياء",
    "الانبياء",
    "اليوم الآخر",
    "اليوم الاخر",
    "الحساب",
    "الجنة",
    "الجنه",
    "النار",
]

fiqh_keywords = [
    # التشريع وأصول الفقه
    "الفقه",
    "الفقة",
    "التشريع",
    "أحكام",
    "احكام",
    "أركان",
    "اركان",
    "المذاهب",
    "الإجماع",
    "الاجماع",
    "السنة",
    "السنه",
    "الجماعة",
    "الجماعه",
    "بدعة",
    "بدعه",
    # العبادات والشعائر
    "عبادة",
    "عباده",
    "الصلاة",
    "الصلاه",
    "الزكاة",
    "الزكاه",
    "الشهادة",
    "الشهاده",
    "الصيام",
    "الحج",
    "الوضوء",
    "الطهارة",
    "الطهاره",
    # المصادر والحديث
    "حديث",
    "قرآن",
    "قرأن",
]

history_keywords = [
    # الشخصيات والتاريخ
    "التاريخ",
    "السيرة",
    "السيره",
    "سيرة",
    "سيره",
    "الصحابة",
    "الصحابه",
    "الخلفاء الراشدين",
    "أتباع",
    "اتباع",
    "الحاكم",
    "عصر",
    # الغزوات والمعارك
    "غزوة",
    "غزوه",
    "معركة",
    "معركه",
    "حرب",
    "الردة",
    "الرده",
    # مفاهيم تاريخية وحوادث
    "استشهاد",
    "تضحية",
    "تضحيه",
    "اجتهاد",
]

In [ ]:
def classify_intent(row):
  intent = ''

  # get the topic
  topic = row.get('source_topics', '')

  # clean it
  topic = str(topic).strip()

  if any(keyword in topic for keyword in aqeedah_keywords):
    intent = 'aqeedah'
  elif any(keyword in topic for keyword in fiqh_keywords):
    intent='fiqh'
  elif any(keyword in topic for keyword in history_keywords):
    intent='history'
  else:
    intent='general'

  return {'intent': intent}

In [ ]:
#@title Map this Function to dataset and save it as a Chcekpoint

dataset = dataset.map(classify_intent)

dataset

In [ ]:
dataset['train'][0]

In [ ]:
dataset_id = "A7med-Ame3/islamic-qa-egyptian"
dataset.push_to_hub(dataset_id, private=True)

### Train the Qwen3 Model

In [ ]:
dataset_id = "A7med-Ame3/islamic-qa-egyptian"

dataset = load_dataset(dataset_id)

dataset

In [ ]:
def formatting_prompt(examples):
  texts = []
  for question, answer in zip(examples['question'], examples['answer']):
    texts.append(f"### السؤال:\n{question}\n### الإجابة:\n{answer}")
  return texts


# def formatting_prompt(examples):
#     texts = []
#     for question, answer in zip(examples['question'], examples['answer']):
#         messages = [
#             {"role": "user", "content": question},
#             {"role": "assistant", "content": answer}
#         ]
#         text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
#         texts.append(text)
#     return texts

In [ ]:
args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    num_train_epochs = 2,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 1,
    optim = "adamw_8bit",
    save_strategy = "steps",
    save_steps = 50,            # Every 50 Steps -> Save a Checkpoint
    weight_decay = 0.001,
    lr_scheduler_type = "linear",
    seed = 3407,
    report_to = "wandb",
    padding_free  = False, # Set to True if > 17 GB VRAM,
    output_dir = 'qwen3-tuned-islamic-qa'
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    formatting_func = formatting_prompt,
    args = args
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# False if it is in the First Time 
# & Set it to True If the training is interrupted at any time and you run trainer.train(resume_from_checkpoint=True) again, 
# it will work normally and will resume from the last checkpoint that was saved!

trainer = trainer.train(resume_from_checkpoint=False)  

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

### Save the fine-tuned model

- After training completes (or if you stop it mid-way when you feel it’s sufficient), save the model.
- This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
# model.save_pretrained("lora_model")  # Local saving
# tokenizer.save_pretrained("lora_model")

# Online saving
model.push_to_hub("A7med-Ame3/qwen3_lora_model", token = HF_TOKEN)
tokenizer.push_to_hub("A7med-Ame3/qwen3_lora_tokenizer", token = HF_TOKEN) 

### Inference

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'A7med-Ame3/qwen3_lora_model',
    max_seq_length=2048,
    load_in_4bit = True
)

In [ ]:
import time

FastLanguageModel.for_inference(model)
question = "ما هى اركان الاسلام ؟"

messages = [{
    'role': 'user',
    'content': question
}]

# prompt = "<|im_start|>" + question + "<|im_end|>" + "<|im_start|>" + "assistant\n"
start_time = time.time()
inputs = tokenizer.apply_chat_template(messages,
                                       tokenize=True,
                                       add_generation_prompt = True, # assistant
                                      return_tensors='pt'
                                ).to(model.device)

outputs = model.generate(inputs,
                        max_new_tokens=128,
                        temperature=0.2,
                        do_sample=True)
end_time = time.time() - start_time

# outputs[0][inputs.shape[1]:] during decode ensures that the model prints only the answer 
# and removes the question text from the output.
output = tokenizer.decode(outputs[0][inputs.shape[1]:],
                         skip_special_tokens=True)

print(f"Inference Time = {end_time}")
print(output)

In [ ]:
output.split("<think>") 

In [ ]:
if "<think>" in output:
    output = output.split("<think>")[-1].strip()

print(output)

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
question = "ما الفرق بين النبي و الرسول ؟"

messages = [{
    'role': 'user',
    'content': question
}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be added for Generation
    return_tensors='pt'
).to(model.device)

_ = model.generate(
    input_ids = inputs,
    max_new_tokens=128,
    temperature=0.2,
    do_sample=True,
    streamer = TextStreamer(tokenizer, skip_prompt=True)
)

#### Measure Some Metrics

1. `TTFT (Time-To-First-Token)`: The time taken from the start of request processing until the model outputs the first token. (A very important metric for real-time streaming).

2. `Total Inference Time`: The total time to generate the complete answer. 

3. `Generated Tokens Count`: The number of new tokens generated by the model.

4. `TPS (Tokens Per Second)`: Generation speed, measured as: $$\text{TPS} = \frac{\text{Generated Tokens Count}}{\text{Total Inference Time}}$$

In [ ]:
class LatencyStreamer(TextStreamer):
    def __init__(self, tokenizer):
        super().__init__(tokenizer, skip_prompt=True)
        self.start_time = None
        self.first_token_time = None
        self.generated_tokens = 0

    def put(self, value):
        current_time = time.time()
        # Record First Time when token generated
        if self.first_token_time is None and value.numel() > 0:
            self.first_token_time = current_time
        
        # Number of Generated Tokens
        self.generated_tokens += value.numel()
        super().put(value)

streamer = LatencyStreamer(tokenizer)

In [ ]:
FastLanguageModel.for_inference(model)
question = "ما الفرق بين النبي و الرسول ؟"

messages = [{
    'role': 'user',
    'content': question
}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be added for Generation
    return_tensors='pt'
).to(model.device)

prompt_length = inputs.shape[1]

start_time = time.time()
streamer.start_time = start_time
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens=128,
    temperature=0.2,
    do_sample=True,
    streamer = TextStreamer(tokenizer, skip_prompt=True)
)
total_time = time.time() - start_time 

ttft = (streamer.first_token_time - start_time) if streamer.first_token_time else total_time
token_count = outputs[0].shape[0] - prompt_length
tps = token_count / total_time if total_time > 0 else 0

output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
if "<think>" in output:
    output = output.split("<think>")[-1].strip()

In [ ]:
print("📊 INFERENCE BENCHMARK REPORT")
print("_"*40)
print(f"⏱️ Time-To-First-Token (TTFT) : {ttft:.4f} sec ({ttft*1000:.2f} ms)")
print(f"⏳ Total Inference Time        : {total_time:.4f} sec")
print(f"🔢 Total Output Tokens        : {token_count} tokens")
print(f"⚡ Tokens Per Second (TPS)     : {tps:.2f} tokens/sec")
print("_"*40)
print("\n📝 Output Answer:\n", output)

### Final Inference Function

In [ ]:
def ask_sheikh(question, max_new_tokens=256, temperature=0.2, repetition_penalty=1.2):
    messages = [
        {
            'role': 'system',
            'content': 'إنت مساعد ذكي بترد على أسئلة دينية بالعامية المصرية بشكل مباشر ومختصر وواضح.'
        },
        {
            'role': 'user',
            'content': question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors='pt'
    ).to(model.device)

    prompt_length = inputs.shape[1]

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        repetition_penalty=repetition_penalty,
        streamer=TextStreamer(tokenizer, skip_prompt=True)
    )

    output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
    return output.strip()

In [ ]:
question = "كم عدد الصلوات فى اليوم الواحد ؟ و لمازا نصوم رمضان ؟"
output = ask_sheikh(question)

print("_"*20)
print(output)

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# model.save_pretrained_gguf(
#     "qwen3-islamic-qa-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )

In [ ]:
# !ollama create qwen3-islamic-qa -f Modelfile
# !ollama run qwen3-islamic-qa

### Merge the Base Model with LoRA Adapter 

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'A7med-Ame3/qwen3_lora_model',
    max_seq_length=2048,
    load_in_4bit = False
)

In [ ]:
model.save_pretrained_merged(
    "qwen3_merged_model",
    tokenizer,
    save_method = "merged_16bit"
)

In [ ]:
model.push_to_hub_merged(
    "A7med-Ame3/qwen3_merged_model", 
    tokenizer, 
    token = HF_TOKEN
)